# 🃏 Poker AI v4 — Google Colab Tréner

**Funkciók:**
- GPU-gyorsított PPO + Self-Play tréning
- Automatikus Google Drive mentés (checkpoint, napló, tesztek)
- Dinamikus mérföldkő intervallum (alapértelmezett: 500 000)
- Azonos mappastruktúra mint a lokális GUI-s futtatás
- Session folytatás: ha a checkpoint már létezik Drive-on, onnan folytat

---
**Futtatás előtt:** Futásidő → GPU módosítása (T4 vagy A100 ajánlott)


## 1. lépés — Google Drive csatolása

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive csatolva: /content/drive/MyDrive')

## 2. lépés — Dependenciák telepítése

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'HIBA ({cmd}):\n{result.stderr[-800:]}')
    else:
        print(f'✅ {cmd}')
    return result.returncode

run('pip install -q rlcard>=1.0.5')
run('pip install -q torch>=2.0.0 --extra-index-url https://download.pytorch.org/whl/cu118')
run('pip install -q tensorboard>=2.13.0')

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA elérhető: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. lépés — Kódbázis feltöltése

Két lehetőség:

**A) GitHub-ról (ha van publikus/privát repo):**
```python
# Cseréld ki a saját repo URL-re
!git clone https://github.com/FELHASZNALONEVED/poker_ai_v4.git /content/poker_ai
```

**B) Drive-on tárolt zip-ből (ajánlott):**
Másold a `poker_ai_v4.zip`-et a Drive-ra (`MyDrive/` alá), majd az alábbi cella futtatja.

In [ ]:
import os, shutil, zipfile

# ── MÓDSZER KIVÁLASZTÁSA ────────────────────────────────────────────────────
# Állítsd be az egyiket True-ra:
USE_DRIVE_ZIP  = True   # Drive-on lévő zip (ajánlott)
USE_GITHUB     = False  # GitHub repo
USE_COLAB_UPLOAD = False  # Közvetlen feltöltés Colab-ba
# ───────────────────────────────────────────────────────────────────────────

CODE_DIR = '/content/poker_ai'

if USE_DRIVE_ZIP:
    # A zip fájl elérési útja a Drive-on
    DRIVE_ZIP_PATH = '/content/drive/MyDrive/poker_ai_v4.zip'

    if not os.path.exists(DRIVE_ZIP_PATH):
        raise FileNotFoundError(
            f'Zip nem található: {DRIVE_ZIP_PATH}\n'
            'Másold a poker_ai_v4.zip-et a Google Drive MyDrive/ mappájába!'
        )

    if os.path.exists(CODE_DIR):
        shutil.rmtree(CODE_DIR)

    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
        z.extractall('/content/poker_ai_raw')

    # A zip tartalmazhat egy belső mappát (pl. 'poker_ai_v4 másolata/')
    # Megkeressük az igazi projekt gyökeret (ahol setup.py vagy train.py van)
    raw_root = '/content/poker_ai_raw'
    found_root = None
    for entry in os.listdir(raw_root):
        candidate = os.path.join(raw_root, entry)
        if os.path.isdir(candidate) and os.path.exists(os.path.join(candidate, 'train.py')):
            found_root = candidate
            break
    if found_root is None and os.path.exists(os.path.join(raw_root, 'train.py')):
        found_root = raw_root

    if found_root is None:
        # Kilistázzuk ami van
        for r, dirs, files in os.walk(raw_root):
            print(r, files[:5])
        raise RuntimeError('Nem találom a train.py-t a zip-ben. Ellenőrizd a struktúrát!')

    shutil.copytree(found_root, CODE_DIR)
    shutil.rmtree(raw_root)
    print(f'✅ Kód kibontva: {CODE_DIR}')

elif USE_GITHUB:
    GITHUB_URL = 'https://github.com/FELHASZNALONEV/poker_ai_v4.git'  # ← CSERE!
    if os.path.exists(CODE_DIR):
        shutil.rmtree(CODE_DIR)
    os.system(f'git clone {GITHUB_URL} {CODE_DIR}')
    print(f'✅ Repo klónozva: {CODE_DIR}')

elif USE_COLAB_UPLOAD:
    from google.colab import files
    print('Válaszd ki a zip fájlt a feltöltési ablakban...')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    if os.path.exists(CODE_DIR):
        shutil.rmtree(CODE_DIR)
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall(CODE_DIR)
    print(f'✅ Feltöltve és kibontva: {CODE_DIR}')

# Projekt gyökér ellenőrzés
assert os.path.exists(os.path.join(CODE_DIR, 'train.py')), \
    f'train.py nem található itt: {CODE_DIR}. Ellenőrizd a zip struktúráját!'
print(f'\nKód struktúra OK → {CODE_DIR}')

## 4. lépés — Módosított fájlok alkalmazása

Ez a cella alkalmazza a Colab-kompatibilis módosításokat a kódbázisra:
- `config.py`: `milestone_hands` mező hozzáadása
- `training/runner.py`: sub-million mérföldkő névgenerálás fix (`500k` nem `0M`)
- `_train_session_cli.py`: `--milestone-interval` és `--drive-output-dir` argumentumok

In [ ]:
import os

CODE_DIR = '/content/poker_ai'

# ══════════════════════════════════════════════════════════════════════════
# PATCH 1: config.py — milestone_hands mező hozzáadása
# ══════════════════════════════════════════════════════════════════════════
config_patch = '''
    milestone_hands: int = 2000
    """[COLAB MOD v1] Sanity teszt kézszám mérföldkőnél.
    Lokálisan: 2000 (default). Colab-on csökkenthető pl. 500-ra."""
'''

config_path = os.path.join(CODE_DIR, 'config.py')
with open(config_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Beillesztjük a milestone_dir_root mező UTÁn
ANCHOR = '    milestone_dir_root: str = "ModellNaplo"'
if 'milestone_hands' not in content:
    # Megkeressük a sor végét és utána adjuk hozzá
    insert_after = content.find(ANCHOR)
    if insert_after != -1:
        # Megkeressük a mező docstringjének végét
        doc_end = content.find('\n\n', insert_after)
        if doc_end == -1:
            doc_end = content.find('\n    # ──', insert_after)
        if doc_end != -1:
            content = content[:doc_end] + '\n' + config_patch + content[doc_end:]
            with open(config_path, 'w', encoding='utf-8') as f:
                f.write(content)
            print('✅ config.py: milestone_hands mező hozzáadva')
        else:
            print('⚠️ config.py: nem találom a beillesztési pontot – kézi ellenőrzés szükséges')
    else:
        print(f'⚠️ config.py: anchor nem található ({ANCHOR[:40]}...) – már módosított?')
else:
    print('ℹ️ config.py: milestone_hands már létezik – patch kihagyva')

# ══════════════════════════════════════════════════════════════════════════
# PATCH 2: training/runner.py — _milestone_str() függvény hozzáadása
# ══════════════════════════════════════════════════════════════════════════
runner_path = os.path.join(CODE_DIR, 'training', 'runner.py')
with open(runner_path, 'r', encoding='utf-8') as f:
    runner = f.read()

MILESTONE_STR_FN = '''
def _milestone_str(milestone_episodes: int) -> str:
    """[COLAB MOD v1] Fix: sub-million intervallumoknál helyesen {N}k nevet ad.
    pl. 500_000 → '500k' (nem '0M')."""
    if milestone_episodes < 1_000_000:
        return f"{milestone_episodes // 1_000}k"
    return f"{milestone_episodes // 1_000_000}M"

'''

if '_milestone_str' not in runner:
    # Hozzáadjuk a _run_milestone előtt
    insert_before = 'def _run_milestone('
    idx = runner.find(insert_before)
    if idx != -1:
        runner = runner[:idx] + MILESTONE_STR_FN + runner[idx:]
        print('✅ runner.py: _milestone_str() függvény hozzáadva')
    else:
        print('⚠️ runner.py: _run_milestone() nem található')
else:
    print('ℹ️ runner.py: _milestone_str() már létezik – patch kihagyva')

# Javítjuk a _run_milestone()-on belüli milestone_m / milestone_str logikát
OLD_MS_LOGIC = (
    'milestone_m = milestone_episodes // 1_000_000\n'
    '    milestone_str = f"{milestone_m}M"'
)
NEW_MS_LOGIC = 'ms_str = _milestone_str(milestone_episodes)'

if OLD_MS_LOGIC in runner:
    runner = runner.replace(OLD_MS_LOGIC, NEW_MS_LOGIC)
    # A továbbiakban milestone_str → ms_str (csak a _run_milestone belsejében)
    # Csak a függvényen belüli milestone_str referenciákat cseréljük
    # Keressük meg a függvény blokkját
    fn_start = runner.find('def _run_milestone(')
    fn_end   = runner.find('\ndef ', fn_start + 1)
    if fn_end == -1:
        fn_end = len(runner)
    fn_block = runner[fn_start:fn_end]
    fn_block_fixed = fn_block.replace(
        'milestone_str', 'ms_str'
    ).replace(
        'ms_str = _milestone_str(milestone_episodes)', 'ms_str = _milestone_str(milestone_episodes)'
    )
    runner = runner[:fn_start] + fn_block_fixed + runner[fn_end:]
    print('✅ runner.py: milestone_str logika javítva (sub-million support)')
elif 'ms_str = _milestone_str' in runner:
    print('ℹ️ runner.py: milestone_str logika már javított – patch kihagyva')
else:
    print('⚠️ runner.py: milestone_str logika nem található – kézi ellenőrzés!')

# milestone_hands: cfg.milestone_hands használata (ha hasattr fallback)
OLD_HANDS = (
    'milestone_hands=cfg.milestone_hands if hasattr(cfg, \'milestone_hands\') '
    'else MILESTONE_HANDS'
)
NEW_HANDS = 'milestone_hands=cfg.milestone_hands'
if OLD_HANDS in runner:
    runner = runner.replace(OLD_HANDS, NEW_HANDS)
    print('✅ runner.py: milestone_hands → cfg.milestone_hands (közvetlen)')

# MILESTONE_DIR_ROOT globál override (a cfg értékével)
GLOBAL_OVERRIDE = '''
    # ── Globál milestone_dir_root beállítása a _run_milestone számára ─────
    global MILESTONE_DIR_ROOT
    MILESTONE_DIR_ROOT = cfg.milestone_dir_root
'''
if 'global MILESTONE_DIR_ROOT' not in runner:
    # A session_start = time.time() sor elé szúrjuk be
    anchor = 'session_start = time.time()'
    idx = runner.find(anchor)
    if idx != -1:
        runner = runner[:idx] + GLOBAL_OVERRIDE + '    ' + runner[idx:]
        print('✅ runner.py: MILESTONE_DIR_ROOT globál override hozzáadva')

with open(runner_path, 'w', encoding='utf-8') as f:
    f.write(runner)

# ══════════════════════════════════════════════════════════════════════════
# PATCH 3: _train_session_cli.py — új argumentumok
# ══════════════════════════════════════════════════════════════════════════
cli_path = os.path.join(CODE_DIR, '_train_session_cli.py')
with open(cli_path, 'r', encoding='utf-8') as f:
    cli = f.read()

NEW_ARGS = '''
    # [COLAB MOD v1] Új argumentumok
    parser.add_argument(
        "--milestone-interval", type=int, default=None,
        help="Felülírja a milestone_interval értékét. Pl. 500000 Colab-on."
    )
    parser.add_argument(
        "--drive-output-dir", default=None,
        help="Külső mentési könyvtár (pl. Google Drive). Ha megadva, ide ment."
    )
'''

DRIVE_LOGIC = '''
    # [COLAB MOD v1] Drive output dir és milestone interval kezelése
    if args.drive_output_dir:
        output_base = args.drive_output_dir
        os.makedirs(output_base, exist_ok=True)
        mgr = ModelManager(output_base)
    else:
        output_base = BASE_DIR

    filtered["milestone_dir_root"] = mgr.tests_dir(args.model_name)
    if args.milestone_interval is not None:
        filtered["milestone_interval"] = args.milestone_interval
        print(f"[COLAB] milestone_interval: {args.milestone_interval:,}")

    mgr.ensure_model_dir(args.model_name, args.players)
'''

if '--milestone-interval' not in cli:
    # Új argparse argumentumok hozzáadása
    anchor = 'args = parser.parse_args()'
    idx = cli.find(anchor)
    if idx != -1:
        cli = cli[:idx] + NEW_ARGS + '\n    ' + cli[idx:]
        print('✅ _train_session_cli.py: --milestone-interval és --drive-output-dir hozzáadva')

    # drive_output_dir logika beillesztése
    anchor2 = 'filtered["milestone_dir_root"] = mgr.tests_dir(args.model_name)'
    idx2 = cli.find(anchor2)
    if idx2 != -1:
        # Cseréljük le az egész részt
        old_block = 'filtered["milestone_dir_root"] = mgr.tests_dir(args.model_name)'
        cli = cli.replace(old_block, DRIVE_LOGIC.strip())
        print('✅ _train_session_cli.py: drive output logika hozzáadva')

    with open(cli_path, 'w', encoding='utf-8') as f:
        f.write(cli)
else:
    print('ℹ️ _train_session_cli.py: argumentumok már léteznek – patch kihagyva')

print('\n✅ Minden patch alkalmazva. Kész a tréningre!')

## 5. lépés — ⚙️ Konfiguráció (ITT ÁLL ÍTS!)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  TRÉNING KONFIGURÁCIÓ — módosítsd az igényeid szerint
# ══════════════════════════════════════════════════════════════════════════

# Modell neve (ez lesz a Drive-on a mappanév is)
MODEL_NAME  = '2max'

# Játékosszám (2 = heads-up, 6 = 6-max, 9 = full ring)
NUM_PLAYERS = 2

# Futtatandó epizódok száma ebben a sessionben
EPISODES    = 2_000_000

# Mérföldkő mentési intervallum (Colab-on ajánlott: 500_000)
# Ennyi epizódonként ment snapshoto + futtat sanity tesztet
MILESTONE_INTERVAL = 500_000

# Sanity teszt kézszám mérföldkőnél
# Csökkentsd 500-ra ha a teszt túl sokáig tart
MILESTONE_HANDS = 2000

# Google Drive mentési könyvtár gyökere
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/PokerAI_Models'

# Tréning stílus: 'exploitative' | 'self_play' | 'aggressive'
TRAINING_STYLE = 'exploitative'

# PPO konfig (leave defaults unless experimenting)
NUM_ENVS            = 256   # GPU VRAM alapján: T4→256, A100→512
BUFFER_COLLECT_SIZE = 2048
HIDDEN_SIZE         = 512
LEARNING_RATE       = 3e-4

# ══════════════════════════════════════════════════════════════════════════
# Számított értékek (ne módosítsd)
# ══════════════════════════════════════════════════════════════════════════
import os

CODE_DIR   = '/content/poker_ai'
MODEL_DIR  = os.path.join(DRIVE_OUTPUT_DIR, 'models', MODEL_NAME)
PTH_PATH   = os.path.join(MODEL_DIR, f'{MODEL_NAME}_ppo_v4.pth')
TESTS_DIR  = os.path.join(MODEL_DIR, 'tests')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TESTS_DIR, exist_ok=True)

print('══════════════════════════════════════════════════')
print('  POKER AI v4 — COLAB TRÉNING KONFIGURÁCIÓ')
print('══════════════════════════════════════════════════')
print(f'  Modell neve:    {MODEL_NAME}')
print(f'  Játékosszám:    {NUM_PLAYERS}')
print(f'  Epizódok:       {EPISODES:,}')
print(f'  Mérföldkő:      minden {MILESTONE_INTERVAL:,} ep')
print(f'  Teszt kéz:      {MILESTONE_HANDS:,}')
print(f'  Stílus:         {TRAINING_STYLE}')
print(f'  Envek:          {NUM_ENVS}')
print('──────────────────────────────────────────────────')
print(f'  Checkpoint:     {PTH_PATH}')
print(f'  Model mappa:    {MODEL_DIR}')
print(f'  Tests mappa:    {TESTS_DIR}')
print('══════════════════════════════════════════════════')

if os.path.exists(PTH_PATH):
    size_mb = os.path.getsize(PTH_PATH) / 1e6
    print(f'\n  ♻️  Meglévő checkpoint találva ({size_mb:.1f} MB) → folytatás innen')
else:
    print('\n  🆕 Nincs meglévő checkpoint → új tréning indul')

## 6. lépés — Ellenőrzés és előkészítés

In [ ]:
import sys, os, json, importlib

# Projekt a Python path-ra
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# ── Import ellenőrzések ──────────────────────────────────────────────────
checks = [
    ('rlcard',                  'RLCard'),
    ('torch',                   'PyTorch'),
    ('config',                  'config.py (projekt)'),
    ('training.model_manager',  'ModelManager'),
    ('training.runner',         'Runner'),
    ('core.model',              'AdvancedPokerAI'),
]

all_ok = True
for module, name in checks:
    try:
        importlib.import_module(module)
        print(f'  ✅ {name}')
    except ImportError as e:
        print(f'  ❌ {name}: {e}')
        all_ok = False

# ── milestone_hands ellenőrzés a config.py-ban ───────────────────────────
from config import TrainingConfig
if hasattr(TrainingConfig, '__dataclass_fields__') and 'milestone_hands' in TrainingConfig.__dataclass_fields__:
    print('  ✅ config.py: milestone_hands mező OK')
else:
    print('  ❌ config.py: milestone_hands mező HIÁNYZIK – futtasd újra a 4. lépést!')
    all_ok = False

# ── GPU ellenőrzés ────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'  ✅ GPU: {torch.cuda.get_device_name(0)}')
    free, total = torch.cuda.mem_get_info()
    print(f'     VRAM szabad: {free/1e9:.1f} GB / {total/1e9:.1f} GB')
    if NUM_ENVS > 256 and total < 12e9:
        print(f'  ⚠️  NUM_ENVS={NUM_ENVS} nagy lehet {total/1e9:.0f}GB VRAM-hoz. '
              f'Csökkentsd 128-ra ha OOM hibát kapsz.')
else:
    print('  ⚠️  Nincs GPU! CPU-n lassú lesz. Futásidő → GPU beállítása ajánlott.')

# ── Drive ellenőrzés ──────────────────────────────────────────────────────
if os.path.isdir('/content/drive/MyDrive'):
    print('  ✅ Google Drive csatolva')
else:
    print('  ❌ Google Drive NEM csatolva! Futtasd az 1. lépést!')
    all_ok = False

# ── ModelManager inicializálás ────────────────────────────────────────────
from training.model_manager import ModelManager
mgr = ModelManager(DRIVE_OUTPUT_DIR)
mgr.ensure_model_dir(MODEL_NAME, NUM_PLAYERS)
print(f'  ✅ Drive modell mappa kész: {MODEL_DIR}')

# config.json összehangolása a Colab konfiggal
cfg_data = mgr.load_config(MODEL_NAME)
cfg_data['config']['milestone_interval']  = MILESTONE_INTERVAL
cfg_data['config']['milestone_hands']     = MILESTONE_HANDS
cfg_data['config']['num_envs']            = NUM_ENVS
cfg_data['config']['buffer_collect_size'] = BUFFER_COLLECT_SIZE
cfg_data['config']['hidden_size']         = HIDDEN_SIZE
cfg_data['config']['learning_rate']       = LEARNING_RATE
cfg_data['config']['training_style']      = TRAINING_STYLE
cfg_data['num_players'] = NUM_PLAYERS
mgr.save_config(MODEL_NAME, cfg_data)
print(f'  ✅ config.json frissítve: {mgr.config_path(MODEL_NAME)}')

print()
if all_ok:
    print('✅ Minden ellenőrzés sikeres — indíthatod a tréninget (7. lépés)')
else:
    print('❌ Hibák vannak — javítsd a pirosakat a tréning előtt!')

## 7. lépés — 🚀 Tréning indítása

> **Megjegyzés:** A Colab session megszakadhat ~12 óra után (ingyenes) vagy ~24 óra után (Pro). A checkpoint automatikusan Drive-ra mentődik minden 1000 epizódonként, és a tréning onnan folytatható.

In [ ]:
import subprocess, sys, os, json

# ── Config JSON összerakás a CLI-hez ─────────────────────────────────────
# A bot_pool stílus preset alapján
STYLE_PRESETS = {
    'exploitative': {
        'fish':            {'enabled': True,  'weight': 0.8},
        'nit':             {'enabled': True,  'weight': 1.5},
        'calling_station': {'enabled': True,  'weight': 0.2},
        'lag':             {'enabled': True,  'weight': 1.5},
    },
    'self_play': {
        'fish':            {'enabled': False, 'weight': 0.0},
        'nit':             {'enabled': False, 'weight': 0.0},
        'calling_station': {'enabled': False, 'weight': 0.0},
        'lag':             {'enabled': False, 'weight': 0.0},
    },
    'aggressive': {
        'fish':            {'enabled': True,  'weight': 0.5},
        'nit':             {'enabled': True,  'weight': 2.5},
        'calling_station': {'enabled': False, 'weight': 0.0},
        'lag':             {'enabled': True,  'weight': 2.5},
    },
}

config_dict = {
    'num_envs':            NUM_ENVS,
    'buffer_collect_size': BUFFER_COLLECT_SIZE,
    'hidden_size':         HIDDEN_SIZE,
    'learning_rate':       LEARNING_RATE,
    'milestone_hands':     MILESTONE_HANDS,
    'training_style':      TRAINING_STYLE,
    'bot_pool':            STYLE_PRESETS.get(TRAINING_STYLE, STYLE_PRESETS['exploitative']),
}

config_json_str = json.dumps(config_dict)

# ── CLI parancs összerakása ───────────────────────────────────────────────
cmd = [
    sys.executable,
    os.path.join(CODE_DIR, '_train_session_cli.py'),
    '--model-name',         MODEL_NAME,
    '--pth-path',           PTH_PATH,
    '--players',            str(NUM_PLAYERS),
    '--episodes',           str(EPISODES),
    '--config-json',        config_json_str,
    '--milestone-interval', str(MILESTONE_INTERVAL),
    '--drive-output-dir',   DRIVE_OUTPUT_DIR,
]

print('Indított parancs:')
print(' '.join(cmd[:6]) + ' ...')
print(f'\nMentési helyek:')
print(f'  Checkpoint:  {PTH_PATH}')
print(f'  Mérföldkövek:{TESTS_DIR}/')
print(f'\nTréning indul...\n')
print('=' * 60)

# Valós idejű output megjelenítés
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, 'PYTHONPATH': CODE_DIR, 'PYTHONUNBUFFERED': '1'}
)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
except KeyboardInterrupt:
    print('\n\n⚠️ Megszakítva! A legutolsó checkpoint Drive-on marad.')
    process.terminate()
    process.wait()

if process.returncode == 0:
    print('\n✅ Tréning sikeresen befejezve!')
elif process.returncode is not None:
    print(f'\n❌ Tréning hiba (kód: {process.returncode})')

## 8. lépés — 📊 Drive struktúra és eredmények ellenőrzése

In [ ]:
import os, json, glob
from datetime import datetime

def format_size(path):
    if not os.path.exists(path): return '(hiányzik)'
    s = os.path.getsize(path)
    if s > 1e6: return f'{s/1e6:.1f} MB'
    if s > 1e3: return f'{s/1e3:.0f} KB'
    return f'{s} B'

print('═══════════════════════════════════════════════════')
print(f'  DRIVE STRUKTÚRA: {MODEL_DIR}')
print('═══════════════════════════════════════════════════')

# Főbb fájlok
files_to_check = [
    (PTH_PATH,                    'Checkpoint (.pth)'),
    (mgr.config_path(MODEL_NAME), 'Konfiguráció (config.json)'),
    (mgr.naplo_path(MODEL_NAME),  'Napló (naplo.json)'),
]

for path, label in files_to_check:
    exists = '✅' if os.path.exists(path) else '❌'
    size   = format_size(path) if os.path.exists(path) else ''
    print(f'  {exists} {label}: {os.path.basename(path)}  {size}')

# Checkpoint meta
if os.path.exists(PTH_PATH):
    try:
        import sys
        sys.path.insert(0, CODE_DIR)
        from utils.checkpoint_utils import safe_load_checkpoint
        ck = safe_load_checkpoint(PTH_PATH, map_location='cpu')
        if isinstance(ck, dict) and 'episodes_trained' in ck:
            ep = ck['episodes_trained']
            t  = ck.get('time_spent', 0)
            print(f'\n  📈 Tréning állapot: {ep:,} epizód | {t/3600:.1f} óra')
    except Exception as e:
        print(f'  ⚠️ Checkpoint meta olvasás sikertelen: {e}')

# Mérföldkövek
print(f'\n  📁 Mérföldkövek ({TESTS_DIR}):')
milestone_dirs = sorted(glob.glob(os.path.join(TESTS_DIR, '*/')), reverse=True)
if not milestone_dirs:
    print('     (még nincs mérföldkő)')
else:
    for md in milestone_dirs[:8]:  # Max 8 legutóbbi
        name = os.path.basename(md.rstrip('/'))
        pth_files   = glob.glob(os.path.join(md, '*.pth'))
        json_files  = glob.glob(os.path.join(md, '*.json'))
        log_files   = glob.glob(os.path.join(md, '*.log'))

        grade = '?'
        if json_files:
            try:
                with open(json_files[0]) as f:
                    jdata = json.load(f)
                grade = jdata.get('summary', {}).get('grade', '?')
            except Exception:
                pass

        pth_ok  = '✅' if pth_files else '❌'
        json_ok = '✅' if json_files else '❌'
        log_ok  = '✅' if log_files else '❌'
        print(f'     📂 {name}  pth:{pth_ok} json:{json_ok} log:{log_ok}  grade:{grade}')

# Napló summary
naplo_path = mgr.naplo_path(MODEL_NAME)
if os.path.exists(naplo_path):
    try:
        with open(naplo_path) as f:
            naplo = json.load(f)
        sessions = naplo.get('sessions', [])
        print(f'\n  📔 Tréning napló: {len(sessions)} session, '
              f'{naplo.get("total_episodes", 0):,} össz-epizód')
    except Exception:
        pass

print('\n═══════════════════════════════════════════════════')

## 9. lépés — ▶️ Tréning folytatása (opcionális)

Ha a session megszakadt, vagy folytatni szeretnéd, egyszerűen futtasd újra a 7. lépést.
A checkpoint automatikusan betöltődik a Drive-ról.

Az alábbi cella csak a folytatás előtti állapotot jeleníti meg:

In [ ]:
import os, sys

sys.path.insert(0, CODE_DIR)
from utils.checkpoint_utils import safe_load_checkpoint

if os.path.exists(PTH_PATH):
    ck = safe_load_checkpoint(PTH_PATH, map_location='cpu')
    if isinstance(ck, dict) and 'episodes_trained' in ck:
        ep     = ck['episodes_trained']
        t_hrs  = ck.get('time_spent', 0) / 3600
        next_ms = ((ep // MILESTONE_INTERVAL) + 1) * MILESTONE_INTERVAL
        print(f'Jelenlegi állapot:')
        print(f'  Epizódok befejezve: {ep:,}')
        print(f'  Tréning idő:        {t_hrs:.1f} óra')
        print(f'  Következő mérföldkő: {next_ms:,} ep')
        print(f'  Még szükséges:      {max(0, next_ms - ep):,} ep')
        print(f'\n→ A 7. lépés futtatása {ep:,}. epizódtól folytatja.')
    else:
        print('Checkpoint megtalálva, de meta nem olvasható.')
else:
    print(f'Nincs checkpoint ezen a helyen: {PTH_PATH}')